In [3]:
# 09_PORTFOLIO_STRESS_REPORTING.IPYNB - COPY/PASTE NOTEBOOK CELLS
# Banking Stress Testing Framework - Python V2 Migration
#
# SAS V1 EQUIVALENCE
# Step 1: Executive ECL summary by Scenario_Name
# Step 2: Summary validation
# Step 3: Scenario ranking by descending Incremental_Stress_Loss
# Step 4: Product-level ECL summary
# Step 5: Risk-segment ECL summary
# Step 6: Top 20 accounts by descending Stressed_ECL
# Step 7: Executive-summary metadata
# Step 8: Final completion message
#
# RUNTIME STANDARD
# Google Drive mount point : /content/gdrive
# Project root             : /content/gdrive/MyDrive/
#                            Banking_Stress_Testing_V2_Python
#
# INSTRUCTIONS
# 1. Create 09_Portfolio_Stress_Reporting.ipynb in Google Colab.
# 2. Create 10 separate Code cells.
# 3. Copy each CELL section below into the corresponding cell.
# 4. Run CELL 01 through CELL 10 strictly in sequence.
# 5. Continue only when each block ends with BLOCK STATUS : SUCCESS.
# =============================================================================


# =============================================================================
# CELL 01 - MODULE HEADER
# =============================================================================

from datetime import datetime

module_start_timestamp = datetime.now()

print("=" * 80)
print("BANKING STRESS TESTING FRAMEWORK")
print("PYTHON V2 MIGRATION")
print("MODULE : 09_PORTFOLIO_STRESS_REPORTING")
print(
    "EXECUTION STARTED :",
    module_start_timestamp.strftime("%Y-%m-%d %H:%M:%S")
)
print("=" * 80)




BANKING STRESS TESTING FRAMEWORK
PYTHON V2 MIGRATION
MODULE : 09_PORTFOLIO_STRESS_REPORTING
EXECUTION STARTED : 2026-10-14 02:33:48


In [4]:
# =============================================================================
# CELL 02 - BLOCK 1: ENVIRONMENT, GDRIVE MOUNT, PATHS, AND LOGGING
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = 0
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 09: PORTFOLIO STRESS REPORTING")
print("BLOCK 1 : ENVIRONMENT, GDRIVE MOUNT, PATHS, AND LOGGING")
print("START TIME :", block_start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)

try:
    from google.colab import drive
    from pathlib import Path
    import logging
    import os
    import sys
    import numpy as np
    import pandas as pd

    MOUNT_POINT = Path("/content/gdrive")

    # Mount only when MyDrive is not already visible in this runtime.
    if not (MOUNT_POINT / "MyDrive").exists():
        # A stale local directory can prevent mounting. Only remove the empty
        # mount directory; never delete a populated directory automatically.
        if MOUNT_POINT.exists() and not any(MOUNT_POINT.iterdir()):
            MOUNT_POINT.rmdir()
        drive.mount(str(MOUNT_POINT))
    else:
        print("Google Drive is already mounted at:", MOUNT_POINT)

    PROJECT_ROOT = (
        MOUNT_POINT
        / "MyDrive"
        / "Banking_Stress_Testing_V2_Python"
    )

    INPUT_FILE = PROJECT_ROOT / "output" / "portfolio_ecl.csv"
    OUTPUT_FILE = PROJECT_ROOT / "output" / "portfolio_stress_report.csv"
    VALIDATION_DIR = PROJECT_ROOT / "validation"
    LOG_DIR = PROJECT_ROOT / "logs"

    ECL_SUMMARY_FILE = VALIDATION_DIR / "09_ecl_summary.csv"
    SCENARIO_RANKING_FILE = VALIDATION_DIR / "09_scenario_ranking.csv"
    PRODUCT_SUMMARY_FILE = VALIDATION_DIR / "09_product_ecl_summary.csv"
    SEGMENT_SUMMARY_FILE = VALIDATION_DIR / "09_segment_ecl_summary.csv"
    TOP_ACCOUNTS_FILE = VALIDATION_DIR / "09_top_20_accounts.csv"
    METADATA_FILE = VALIDATION_DIR / "09_executive_summary_metadata.csv"
    VALIDATION_FILE = VALIDATION_DIR / "09_portfolio_reporting_validation.csv"
    SAMPLE_FILE = VALIDATION_DIR / "09_portfolio_reporting_sample.csv"
    LOG_FILE = LOG_DIR / "09_Portfolio_Stress_Reporting.log"

    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
    LOG_DIR.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger("09_Portfolio_Stress_Reporting")
    logger.setLevel(logging.INFO)
    if logger.hasHandlers():
        logger.handlers.clear()

    log_formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )
    file_handler = logging.FileHandler(
        LOG_FILE,
        mode="w",
        encoding="utf-8"
    )
    file_handler.setFormatter(log_formatter)
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(log_formatter)
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

    if not PROJECT_ROOT.exists():
        raise FileNotFoundError(
            f"Project root was not found: {PROJECT_ROOT}"
        )
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"Module 08 output was not found: {INPUT_FILE}"
        )

    print("Drive Mount Exists:", MOUNT_POINT.exists())
    print("Project Root Exists:", PROJECT_ROOT.exists())
    print("Module 08 Output Exists:", INPUT_FILE.exists())
    print("Reporting Input:", INPUT_FILE)
    print("Reporting Output:", OUTPUT_FILE)
    print("Validation Directory:", VALIDATION_DIR)
    print("Execution Log:", LOG_FILE)

    logger.info(
        "Environment, gdrive mount, paths, and logging configured successfully."
    )
    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"
    if "logger" in locals():
        logger.exception(
            "Block 1 failed during environment configuration."
        )
    raise

finally:
    block_end_time = datetime.now()
    block_duration_seconds = round(
        (block_end_time - block_start_time).total_seconds(),
        3
    )
    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print("END TIME           :", block_end_time.strftime("%Y-%m-%d %H:%M:%S"))
    print("DURATION (SECONDS) :", block_duration_seconds)
    print("=" * 80)




MODULE 09: PORTFOLIO STRESS REPORTING
BLOCK 1 : ENVIRONMENT, GDRIVE MOUNT, PATHS, AND LOGGING
START TIME : 2026-10-14 02:35:06
Google Drive is already mounted at: /content/gdrive
Drive Mount Exists: True
Project Root Exists: True
Module 08 Output Exists: True
Reporting Input: /content/gdrive/MyDrive/Banking_Stress_Testing_V2_Python/output/portfolio_ecl.csv
Reporting Output: /content/gdrive/MyDrive/Banking_Stress_Testing_V2_Python/output/portfolio_stress_report.csv
Validation Directory: /content/gdrive/MyDrive/Banking_Stress_Testing_V2_Python/validation
Execution Log: /content/gdrive/MyDrive/Banking_Stress_Testing_V2_Python/logs/09_Portfolio_Stress_Reporting.log
2026-10-14 02:35:07,673 | INFO | Environment, gdrive mount, paths, and logging configured successfully.


INFO:09_Portfolio_Stress_Reporting:Environment, gdrive mount, paths, and logging configured successfully.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 0
OUTPUT RECORDS     : 0
RECORDS PROCESSED  : 0
END TIME           : 2026-10-14 02:35:07
DURATION (SECONDS) : 1.249


In [5]:
# =============================================================================
# CELL 03 - BLOCK 2: LOAD AND VALIDATE MODULE 08 OUTPUT
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = 0
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 09: PORTFOLIO STRESS REPORTING")
print("BLOCK 2 : LOAD AND VALIDATE MODULE 08 OUTPUT")
print("START TIME :", block_start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)

try:
    required_columns = [
        "Customer_ID",
        "Loan_ID",
        "Scenario_ID",
        "Scenario_Name",
        "Product_Type",
        "Risk_Segment",
        "EAD",
        "Base_ECL",
        "Stressed_ECL",
        "Incremental_Stress_Loss",
    ]

    portfolio_ecl = pd.read_csv(
        INPUT_FILE,
        low_memory=False
    )

    input_record_count = len(portfolio_ecl)
    input_column_count = len(portfolio_ecl.columns)

    missing_required_columns = sorted(
        set(required_columns) - set(portfolio_ecl.columns)
    )
    duplicate_loan_scenario_keys = int(
        portfolio_ecl[["Loan_ID", "Scenario_ID"]]
        .duplicated()
        .sum()
    )

    if missing_required_columns:
        raise ValueError(
            f"Required reporting columns are missing: {missing_required_columns}"
        )

    missing_required_values = int(
        portfolio_ecl[required_columns]
        .isna()
        .sum()
        .sum()
    )

    numeric_columns = [
        "EAD",
        "Base_ECL",
        "Stressed_ECL",
        "Incremental_Stress_Loss",
    ]
    for column in numeric_columns:
        portfolio_ecl[column] = pd.to_numeric(
            portfolio_ecl[column],
            errors="raise"
        )

    negative_ead = int((portfolio_ecl["EAD"] < 0).sum())
    negative_base_ecl = int((portfolio_ecl["Base_ECL"] < 0).sum())
    negative_stressed_ecl = int((portfolio_ecl["Stressed_ECL"] < 0).sum())

    scenario_counts = (
        portfolio_ecl["Scenario_Name"]
        .value_counts(dropna=False)
    )

    expected_scenarios = {
        "Baseline",
        "Mild_Stress",
        "Severe_Stress",
    }
    actual_scenarios = set(
        portfolio_ecl["Scenario_Name"]
        .dropna()
        .unique()
    )
    unexpected_scenarios = sorted(
        actual_scenarios - expected_scenarios
    )
    missing_scenarios = sorted(
        expected_scenarios - actual_scenarios
    )

    if input_record_count != 30000:
        raise ValueError(
            "Module 08 input does not contain the expected 30,000 records."
        )
    if duplicate_loan_scenario_keys > 0:
        raise ValueError(
            "Duplicate Loan_ID and Scenario_ID combinations were found."
        )
    if missing_required_values > 0:
        raise ValueError(
            "Missing values were found in required reporting fields."
        )
    if negative_ead > 0 or negative_base_ecl > 0 or negative_stressed_ecl > 0:
        raise ValueError(
            "Negative exposure or ECL values were found."
        )
    if unexpected_scenarios or missing_scenarios:
        raise ValueError(
            "Scenario validation failed. "
            f"Unexpected={unexpected_scenarios}, Missing={missing_scenarios}"
        )
    if not (scenario_counts == 10000).all():
        raise ValueError(
            "Each scenario must contain exactly 10,000 records."
        )

    records_processed = input_record_count

    print("=" * 80)
    print("MODULE 08 OUTPUT LOADED SUCCESSFULLY")
    print("=" * 80)
    print("Input Records:", input_record_count)
    print("Input Columns:", input_column_count)
    print("Input File Size:", INPUT_FILE.stat().st_size, "bytes")
    print("Missing Required Columns:", missing_required_columns)
    print("Duplicate Loan-Scenario Keys:", duplicate_loan_scenario_keys)
    print("Missing Required Values:", missing_required_values)
    print("Negative EAD Values:", negative_ead)
    print("Negative Base ECL Values:", negative_base_ecl)
    print("Negative Stressed ECL Values:", negative_stressed_ecl)
    print("Unexpected Scenarios:", unexpected_scenarios)
    print("Missing Scenarios:", missing_scenarios)
    print("Input Validation Status: PASS")

    print("\nScenario Record Counts:")
    display(
        scenario_counts
        .rename_axis("Scenario_Name")
        .reset_index(name="Record_Count")
    )

    display(
        portfolio_ecl[required_columns]
        .head(15)
    )

    logger.info(
        "Module 08 output loaded successfully: %s records and %s columns.",
        input_record_count,
        input_column_count
    )
    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"
    logger.exception(
        "Block 2 failed while loading Module 08 output."
    )
    raise

finally:
    block_end_time = datetime.now()
    block_duration_seconds = round(
        (block_end_time - block_start_time).total_seconds(),
        3
    )
    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print("END TIME           :", block_end_time.strftime("%Y-%m-%d %H:%M:%S"))
    print("DURATION (SECONDS) :", block_duration_seconds)
    print("=" * 80)




MODULE 09: PORTFOLIO STRESS REPORTING
BLOCK 2 : LOAD AND VALIDATE MODULE 08 OUTPUT
START TIME : 2026-10-14 02:36:32
MODULE 08 OUTPUT LOADED SUCCESSFULLY
Input Records: 30000
Input Columns: 42
Input File Size: 9293904 bytes
Missing Required Columns: []
Duplicate Loan-Scenario Keys: 0
Missing Required Values: 0
Negative EAD Values: 0
Negative Base ECL Values: 0
Negative Stressed ECL Values: 0
Unexpected Scenarios: []
Missing Scenarios: []
Input Validation Status: PASS

Scenario Record Counts:


,Scenario_Name,Record_Count
0,Baseline,10000
1,Mild_Stress,10000
2,Severe_Stress,10000


,Customer_ID,Loan_ID,Scenario_ID,Scenario_Name,Product_Type,Risk_Segment,EAD,Base_ECL,Stressed_ECL,Incremental_Stress_Loss
0,1000001,50000001,1,Baseline,Personal Loan,Low Risk,1733362.19,20800.346280,2.080035e+04,0.000000
1,1000001,50000001,2,Mild_Stress,Personal Loan,Low Risk,1733362.19,20800.346280,2.860048e+04,7800.129855
2,1000001,50000001,3,Severe_Stress,Personal Loan,Low Risk,1733362.19,20800.346280,4.550076e+04,24700.411208
3,1000002,50000002,1,Baseline,Corporate Loan,Low Risk,62244073.93,560196.665370,5.601967e+05,0.000000
4,1000002,50000002,2,Mild_Stress,Corporate Loan,Low Risk,62244073.93,560196.665370,7.702704e+05,210073.749514
5,1000002,50000002,3,Severe_Stress,Corporate Loan,Low Risk,62244073.93,560196.665370,1.225430e+06,665233.540127
6,1000003,50000003,1,Baseline,Credit Card,High Risk,303259.15,25777.027750,2.577703e+04,0.000000
7,1000003,50000003,2,Mild_Stress,Credit Card,High Risk,303259.15,25777.027750,3.544341e+04,9666.385406
8,1000003,50000003,3,Severe_Stress,Credit Card,High Risk,303259.15,25777.027750,5.307035e+04,27293.323500
9,1000004,50000004,1,Baseline,Auto Loan,Medium Risk,993246.23,17381.809025,1.738181e+04,0.000000


2026-10-14 02:36:33,889 | INFO | Module 08 output loaded successfully: 30000 records and 42 columns.


INFO:09_Portfolio_Stress_Reporting:Module 08 output loaded successfully: 30000 records and 42 columns.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 30000
OUTPUT RECORDS     : 0
RECORDS PROCESSED  : 30000
END TIME           : 2026-10-14 02:36:33
DURATION (SECONDS) : 1.098


In [6]:
# =============================================================================
# CELL 04 - BLOCK 3: CREATE EXECUTIVE ECL SUMMARY
# SAS V1: STRESS.ECL_SUMMARY
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = len(portfolio_ecl)
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 09: PORTFOLIO STRESS REPORTING")
print("BLOCK 3 : CREATE EXECUTIVE ECL SUMMARY")
print("START TIME :", block_start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)

try:
    ecl_summary = (
        portfolio_ecl
        .groupby(
            "Scenario_Name",
            as_index=False,
            sort=True
        )
        .agg(
            Account_Count=("Loan_ID", "size"),
            Total_Exposure=("EAD", "sum"),
            Total_Base_ECL=("Base_ECL", "sum"),
            Total_Stressed_ECL=("Stressed_ECL", "sum"),
            Incremental_Stress_Loss=("Incremental_Stress_Loss", "sum"),
        )
    )

    expected_summary_rows = 3
    executive_account_total = int(
        ecl_summary["Account_Count"].sum()
    )
    executive_exposure_total = float(
        ecl_summary["Total_Exposure"].sum()
    )
    source_exposure_total = float(
        portfolio_ecl["EAD"].sum()
    )
    executive_base_ecl_total = float(
        ecl_summary["Total_Base_ECL"].sum()
    )
    source_base_ecl_total = float(
        portfolio_ecl["Base_ECL"].sum()
    )
    executive_stressed_ecl_total = float(
        ecl_summary["Total_Stressed_ECL"].sum()
    )
    source_stressed_ecl_total = float(
        portfolio_ecl["Stressed_ECL"].sum()
    )
    executive_incremental_total = float(
        ecl_summary["Incremental_Stress_Loss"].sum()
    )
    source_incremental_total = float(
        portfolio_ecl["Incremental_Stress_Loss"].sum()
    )

    executive_summary_exceptions = 0
    executive_summary_exceptions += int(
        len(ecl_summary) != expected_summary_rows
    )
    executive_summary_exceptions += int(
        executive_account_total != input_record_count
    )
    executive_summary_exceptions += int(
        not np.isclose(executive_exposure_total, source_exposure_total)
    )
    executive_summary_exceptions += int(
        not np.isclose(executive_base_ecl_total, source_base_ecl_total)
    )
    executive_summary_exceptions += int(
        not np.isclose(
            executive_stressed_ecl_total,
            source_stressed_ecl_total
        )
    )
    executive_summary_exceptions += int(
        not np.isclose(
            executive_incremental_total,
            source_incremental_total
        )
    )

    if executive_summary_exceptions > 0:
        raise ValueError(
            "Executive ECL summary reconciliation failed."
        )

    output_record_count = len(ecl_summary)
    records_processed = input_record_count

    print("=" * 80)
    print("EXECUTIVE STRESS TESTING SUMMARY")
    print("=" * 80)
    display(ecl_summary.round(2))
    print("Executive Summary Rows:", output_record_count)
    print("Executive Account Total:", executive_account_total)
    print("Executive Summary Exceptions:", executive_summary_exceptions)
    print("Executive Summary Validation Status: PASS")

    logger.info(
        "Executive ECL summary generated and reconciled successfully."
    )
    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"
    logger.exception(
        "Block 3 failed while creating executive ECL summary."
    )
    raise

finally:
    block_end_time = datetime.now()
    block_duration_seconds = round(
        (block_end_time - block_start_time).total_seconds(),
        3
    )
    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print("END TIME           :", block_end_time.strftime("%Y-%m-%d %H:%M:%S"))
    print("DURATION (SECONDS) :", block_duration_seconds)
    print("=" * 80)




MODULE 09: PORTFOLIO STRESS REPORTING
BLOCK 3 : CREATE EXECUTIVE ECL SUMMARY
START TIME : 2026-10-14 02:38:22
EXECUTIVE STRESS TESTING SUMMARY


,Scenario_Name,Account_Count,Total_Exposure,Total_Base_ECL,Total_Stressed_ECL,Incremental_Stress_Loss
0,Baseline,10000,4.758191e+10,9.299713e+08,9.299713e+08,0.000000e+00
1,Mild_Stress,10000,4.758191e+10,9.299713e+08,1.278711e+09,3.487392e+08
2,Severe_Stress,10000,4.758191e+10,9.299713e+08,2.031954e+09,1.101983e+09


Executive Summary Rows: 3
Executive Account Total: 30000
Executive Summary Exceptions: 0
Executive Summary Validation Status: PASS
2026-10-14 02:38:22,618 | INFO | Executive ECL summary generated and reconciled successfully.


INFO:09_Portfolio_Stress_Reporting:Executive ECL summary generated and reconciled successfully.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 30000
OUTPUT RECORDS     : 3
RECORDS PROCESSED  : 30000
END TIME           : 2026-10-14 02:38:22
DURATION (SECONDS) : 0.049


In [7]:
# =============================================================================
# CELL 05 - BLOCK 4: CREATE SCENARIO RANKING
# SAS V1: SORT BY DESCENDING INCREMENTAL_STRESS_LOSS
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = len(ecl_summary)
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 09: PORTFOLIO STRESS REPORTING")
print("BLOCK 4 : CREATE SCENARIO RANKING")
print("START TIME :", block_start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)

try:
    scenario_ranking = (
        ecl_summary
        .sort_values(
            by="Incremental_Stress_Loss",
            ascending=False,
            kind="stable"
        )
        .reset_index(drop=True)
    )
    scenario_ranking.insert(
        0,
        "Scenario_Rank",
        range(1, len(scenario_ranking) + 1)
    )

    output_record_count = len(scenario_ranking)
    records_processed = input_record_count

    ranking_order_valid = bool(
        scenario_ranking["Incremental_Stress_Loss"]
        .is_monotonic_decreasing
    )
    ranking_scenarios_match = (
        set(scenario_ranking["Scenario_Name"])
        == expected_scenarios
    )

    if output_record_count != 3:
        raise ValueError(
            "Scenario ranking does not contain three scenarios."
        )
    if not ranking_order_valid:
        raise ValueError(
            "Scenario ranking is not in descending stress-loss order."
        )
    if not ranking_scenarios_match:
        raise ValueError(
            "Scenario ranking does not contain the expected scenarios."
        )

    print("=" * 80)
    print("SCENARIO RANKING BY STRESS LOSS")
    print("=" * 80)
    display(scenario_ranking.round(2))
    print("Scenario Ranking Count:", output_record_count)
    print("Descending Order Valid:", ranking_order_valid)
    print("Expected Scenarios Match:", ranking_scenarios_match)
    print("Scenario Ranking Validation Status: PASS")

    logger.info(
        "Scenario ranking generated successfully."
    )
    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"
    logger.exception(
        "Block 4 failed while creating scenario ranking."
    )
    raise

finally:
    block_end_time = datetime.now()
    block_duration_seconds = round(
        (block_end_time - block_start_time).total_seconds(),
        3
    )
    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print("END TIME           :", block_end_time.strftime("%Y-%m-%d %H:%M:%S"))
    print("DURATION (SECONDS) :", block_duration_seconds)
    print("=" * 80)




MODULE 09: PORTFOLIO STRESS REPORTING
BLOCK 4 : CREATE SCENARIO RANKING
START TIME : 2026-10-14 02:39:21
SCENARIO RANKING BY STRESS LOSS


,Scenario_Rank,Scenario_Name,Account_Count,Total_Exposure,Total_Base_ECL,Total_Stressed_ECL,Incremental_Stress_Loss
0,1,Severe_Stress,10000,4.758191e+10,9.299713e+08,2.031954e+09,1.101983e+09
1,2,Mild_Stress,10000,4.758191e+10,9.299713e+08,1.278711e+09,3.487392e+08
2,3,Baseline,10000,4.758191e+10,9.299713e+08,9.299713e+08,0.000000e+00


Scenario Ranking Count: 3
Descending Order Valid: True
Expected Scenarios Match: True
Scenario Ranking Validation Status: PASS
2026-10-14 02:39:21,318 | INFO | Scenario ranking generated successfully.


INFO:09_Portfolio_Stress_Reporting:Scenario ranking generated successfully.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 3
OUTPUT RECORDS     : 3
RECORDS PROCESSED  : 3
END TIME           : 2026-10-14 02:39:21
DURATION (SECONDS) : 0.022


In [8]:

# =============================================================================
# CELL 06 - BLOCK 5: CREATE PRODUCT-LEVEL ECL SUMMARY
# SAS V1: STRESS.PRODUCT_ECL_SUMMARY
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = len(portfolio_ecl)
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 09: PORTFOLIO STRESS REPORTING")
print("BLOCK 5 : CREATE PRODUCT-LEVEL ECL SUMMARY")
print("START TIME :", block_start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)

try:
    product_ecl_summary = (
        portfolio_ecl
        .groupby(
            ["Scenario_Name", "Product_Type"],
            as_index=False,
            sort=True
        )
        .agg(
            Account_Count=("Loan_ID", "size"),
            Total_Exposure=("EAD", "sum"),
            Total_Stressed_ECL=("Stressed_ECL", "sum"),
        )
    )

    product_account_total = int(
        product_ecl_summary["Account_Count"].sum()
    )
    product_exposure_total = float(
        product_ecl_summary["Total_Exposure"].sum()
    )
    product_stressed_ecl_total = float(
        product_ecl_summary["Total_Stressed_ECL"].sum()
    )

    product_summary_exceptions = 0
    product_summary_exceptions += int(
        product_account_total != input_record_count
    )
    product_summary_exceptions += int(
        not np.isclose(product_exposure_total, source_exposure_total)
    )
    product_summary_exceptions += int(
        not np.isclose(
            product_stressed_ecl_total,
            source_stressed_ecl_total
        )
    )

    if product_summary_exceptions > 0:
        raise ValueError(
            "Product-level ECL summary reconciliation failed."
        )

    output_record_count = len(product_ecl_summary)
    records_processed = input_record_count

    print("=" * 80)
    print("PRODUCT LEVEL STRESS REPORT")
    print("=" * 80)
    display(product_ecl_summary.head(50).round(2))
    print("Product Summary Rows:", output_record_count)
    print("Product Account Total:", product_account_total)
    print("Product Summary Exceptions:", product_summary_exceptions)
    print("Product Summary Validation Status: PASS")

    logger.info(
        "Product-level ECL summary generated and reconciled successfully."
    )
    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"
    logger.exception(
        "Block 5 failed while creating product ECL summary."
    )
    raise

finally:
    block_end_time = datetime.now()
    block_duration_seconds = round(
        (block_end_time - block_start_time).total_seconds(),
        3
    )
    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print("END TIME           :", block_end_time.strftime("%Y-%m-%d %H:%M:%S"))
    print("DURATION (SECONDS) :", block_duration_seconds)
    print("=" * 80)



MODULE 09: PORTFOLIO STRESS REPORTING
BLOCK 5 : CREATE PRODUCT-LEVEL ECL SUMMARY
START TIME : 2026-10-14 02:40:37
PRODUCT LEVEL STRESS REPORT


,Scenario_Name,Product_Type,Account_Count,Total_Exposure,Total_Stressed_ECL
0,Baseline,Auto Loan,2088,3.782912e+09,7.183463e+07
1,Baseline,Corporate Loan,405,1.581266e+10,3.841704e+08
2,Baseline,Credit Card,1555,3.931793e+08,1.832473e+07
3,Baseline,Mortgage,2550,1.594025e+10,1.736888e+08
4,Baseline,Personal Loan,2117,2.170796e+09,7.134425e+07
5,Baseline,SME Loan,1285,9.482109e+09,2.106084e+08
6,Mild_Stress,Auto Loan,2088,3.782912e+09,9.877262e+07
7,Mild_Stress,Corporate Loan,405,1.581266e+10,5.282344e+08
8,Mild_Stress,Credit Card,1555,3.931793e+08,2.519651e+07
9,Mild_Stress,Mortgage,2550,1.594025e+10,2.388221e+08


Product Summary Rows: 18
Product Account Total: 30000
Product Summary Exceptions: 0
Product Summary Validation Status: PASS
2026-10-14 02:40:37,736 | INFO | Product-level ECL summary generated and reconciled successfully.


INFO:09_Portfolio_Stress_Reporting:Product-level ECL summary generated and reconciled successfully.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 30000
OUTPUT RECORDS     : 18
RECORDS PROCESSED  : 30000
END TIME           : 2026-10-14 02:40:37
DURATION (SECONDS) : 0.033


In [9]:
# =============================================================================
# CELL 07 - BLOCK 6: CREATE RISK-SEGMENT ECL SUMMARY
# SAS V1: STRESS.SEGMENT_ECL_SUMMARY
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = len(portfolio_ecl)
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 09: PORTFOLIO STRESS REPORTING")
print("BLOCK 6 : CREATE RISK-SEGMENT ECL SUMMARY")
print("START TIME :", block_start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)

try:
    segment_ecl_summary = (
        portfolio_ecl
        .groupby(
            ["Scenario_Name", "Risk_Segment"],
            as_index=False,
            sort=True
        )
        .agg(
            Account_Count=("Loan_ID", "size"),
            Total_Exposure=("EAD", "sum"),
            Total_Stressed_ECL=("Stressed_ECL", "sum"),
        )
    )

    segment_account_total = int(
        segment_ecl_summary["Account_Count"].sum()
    )
    segment_exposure_total = float(
        segment_ecl_summary["Total_Exposure"].sum()
    )
    segment_stressed_ecl_total = float(
        segment_ecl_summary["Total_Stressed_ECL"].sum()
    )

    segment_summary_exceptions = 0
    segment_summary_exceptions += int(
        segment_account_total != input_record_count
    )
    segment_summary_exceptions += int(
        not np.isclose(segment_exposure_total, source_exposure_total)
    )
    segment_summary_exceptions += int(
        not np.isclose(
            segment_stressed_ecl_total,
            source_stressed_ecl_total
        )
    )

    if segment_summary_exceptions > 0:
        raise ValueError(
            "Risk-segment ECL summary reconciliation failed."
        )

    output_record_count = len(segment_ecl_summary)
    records_processed = input_record_count

    print("=" * 80)
    print("RISK SEGMENT STRESS REPORT")
    print("=" * 80)
    display(segment_ecl_summary.round(2))
    print("Segment Summary Rows:", output_record_count)
    print("Segment Account Total:", segment_account_total)
    print("Segment Summary Exceptions:", segment_summary_exceptions)
    print("Segment Summary Validation Status: PASS")

    logger.info(
        "Risk-segment ECL summary generated and reconciled successfully."
    )
    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"
    logger.exception(
        "Block 6 failed while creating segment ECL summary."
    )
    raise

finally:
    block_end_time = datetime.now()
    block_duration_seconds = round(
        (block_end_time - block_start_time).total_seconds(),
        3
    )
    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print("END TIME           :", block_end_time.strftime("%Y-%m-%d %H:%M:%S"))
    print("DURATION (SECONDS) :", block_duration_seconds)
    print("=" * 80)




MODULE 09: PORTFOLIO STRESS REPORTING
BLOCK 6 : CREATE RISK-SEGMENT ECL SUMMARY
START TIME : 2026-10-14 02:41:51
RISK SEGMENT STRESS REPORT


,Scenario_Name,Risk_Segment,Account_Count,Total_Exposure,Total_Stressed_ECL
0,Baseline,High Risk,2485,1.172251e+10,4.214570e+08
1,Baseline,Low Risk,2535,1.236121e+10,8.908115e+07
2,Baseline,Medium Risk,4980,2.349819e+10,4.194331e+08
3,Mild_Stress,High Risk,2485,1.172251e+10,5.795034e+08
4,Mild_Stress,Low Risk,2535,1.236121e+10,1.224866e+08
5,Mild_Stress,Medium Risk,4980,2.349819e+10,5.767205e+08
6,Severe_Stress,High Risk,2485,1.172251e+10,9.208785e+08
7,Severe_Stress,Low Risk,2535,1.236121e+10,1.946506e+08
8,Severe_Stress,Medium Risk,4980,2.349819e+10,9.164251e+08


Segment Summary Rows: 9
Segment Account Total: 30000
Segment Summary Exceptions: 0
Segment Summary Validation Status: PASS
2026-10-14 02:41:51,033 | INFO | Risk-segment ECL summary generated and reconciled successfully.


INFO:09_Portfolio_Stress_Reporting:Risk-segment ECL summary generated and reconciled successfully.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 30000
OUTPUT RECORDS     : 9
RECORDS PROCESSED  : 30000
END TIME           : 2026-10-14 02:41:51
DURATION (SECONDS) : 0.03


In [10]:
# =============================================================================
# CELL 08 - BLOCK 7: IDENTIFY TOP 20 ACCOUNTS BY STRESSED ECL
# SAS V1: WORK.TOP_ACCOUNTS, OBS=20
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = len(portfolio_ecl)
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 09: PORTFOLIO STRESS REPORTING")
print("BLOCK 7 : IDENTIFY TOP 20 ACCOUNTS BY STRESSED ECL")
print("START TIME :", block_start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)

try:
    top_20_accounts = (
        portfolio_ecl
        .sort_values(
            by="Stressed_ECL",
            ascending=False,
            kind="stable"
        )
        .head(20)
        .reset_index(drop=True)
    )
    top_20_accounts.insert(
        0,
        "Stress_Rank",
        range(1, len(top_20_accounts) + 1)
    )

    output_record_count = len(top_20_accounts)
    records_processed = input_record_count
    top_20_order_valid = bool(
        top_20_accounts["Stressed_ECL"]
        .is_monotonic_decreasing
    )
    top_20_missing_values = int(
        top_20_accounts[
            [
                "Loan_ID",
                "Scenario_Name",
                "EAD",
                "Stressed_ECL",
            ]
        ]
        .isna()
        .sum()
        .sum()
    )

    if output_record_count != 20:
        raise ValueError(
            "Top-account report does not contain 20 records."
        )
    if not top_20_order_valid:
        raise ValueError(
            "Top accounts are not sorted by descending Stressed_ECL."
        )
    if top_20_missing_values > 0:
        raise ValueError(
            "Top-account report contains missing required values."
        )

    print("=" * 80)
    print("TOP 20 ACCOUNTS BY STRESSED ECL")
    print("=" * 80)
    display(
        top_20_accounts[
            [
                "Stress_Rank",
                "Customer_ID",
                "Loan_ID",
                "Scenario_Name",
                "Product_Type",
                "Risk_Segment",
                "EAD",
                "Base_ECL",
                "Stressed_ECL",
                "Incremental_Stress_Loss",
            ]
        ]
        .round(2)
    )
    print("Top Account Count:", output_record_count)
    print("Descending Order Valid:", top_20_order_valid)
    print("Missing Required Values:", top_20_missing_values)
    print("Top 20 Account Validation Status: PASS")

    logger.info(
        "Top 20 stressed accounts identified successfully."
    )
    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"
    logger.exception(
        "Block 7 failed while identifying top stressed accounts."
    )
    raise

finally:
    block_end_time = datetime.now()
    block_duration_seconds = round(
        (block_end_time - block_start_time).total_seconds(),
        3
    )
    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print("END TIME           :", block_end_time.strftime("%Y-%m-%d %H:%M:%S"))
    print("DURATION (SECONDS) :", block_duration_seconds)
    print("=" * 80)




MODULE 09: PORTFOLIO STRESS REPORTING
BLOCK 7 : IDENTIFY TOP 20 ACCOUNTS BY STRESSED ECL
START TIME : 2026-10-14 02:43:10
TOP 20 ACCOUNTS BY STRESSED ECL


,Stress_Rank,Customer_ID,Loan_ID,Scenario_Name,Product_Type,Risk_Segment,EAD,Base_ECL,Stressed_ECL,Incremental_Stress_Loss
0,1,1005503,50005503,Severe_Stress,Corporate Loan,High Risk,74351936.90,3345837.16,7319018.79,3973181.63
1,2,1000833,50000833,Severe_Stress,Corporate Loan,High Risk,72412384.54,3258557.30,7128094.10,3869536.80
2,3,1006773,50006773,Severe_Stress,Corporate Loan,High Risk,72344691.07,3255511.10,7121430.53,3865919.43
3,4,1004328,50004328,Severe_Stress,Corporate Loan,High Risk,70788686.34,3185490.89,6968261.31,3782770.43
4,5,1006883,50006883,Severe_Stress,Corporate Loan,High Risk,70447035.57,3170116.60,6934630.06,3764513.46
5,6,1009658,50009658,Severe_Stress,Corporate Loan,High Risk,68482049.28,3081692.22,6741201.73,3659509.51
6,7,1007000,50007000,Severe_Stress,Corporate Loan,High Risk,67866892.66,3054010.17,6680647.25,3626637.08
7,8,1001543,50001543,Severe_Stress,Corporate Loan,High Risk,67637225.55,3043675.15,6658039.39,3614364.24
8,9,1009705,50009705,Severe_Stress,Corporate Loan,High Risk,65388066.70,2942463.00,6436637.82,3494174.81
9,10,1006633,50006633,Severe_Stress,Corporate Loan,High Risk,65316108.72,2939224.89,6429554.45,3490329.56


Top Account Count: 20
Descending Order Valid: True
Missing Required Values: 0
Top 20 Account Validation Status: PASS
2026-10-14 02:43:10,780 | INFO | Top 20 stressed accounts identified successfully.


INFO:09_Portfolio_Stress_Reporting:Top 20 stressed accounts identified successfully.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 30000
OUTPUT RECORDS     : 20
RECORDS PROCESSED  : 30000
END TIME           : 2026-10-14 02:43:10
DURATION (SECONDS) : 0.049


In [11]:
# =============================================================================
# CELL 09 - BLOCK 8: METADATA, FINAL VALIDATION, AND RECONCILIATION
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = len(portfolio_ecl)
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 09: PORTFOLIO STRESS REPORTING")
print("BLOCK 8 : METADATA, FINAL VALIDATION, AND RECONCILIATION")
print("START TIME :", block_start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)

try:
    executive_summary_metadata = pd.DataFrame({
        "Column_Name": ecl_summary.columns,
        "Python_Data_Type": [
            str(data_type)
            for data_type in ecl_summary.dtypes
        ],
        "Non_Missing_Count": [
            int(ecl_summary[column].notna().sum())
            for column in ecl_summary.columns
        ],
        "Missing_Count": [
            int(ecl_summary[column].isna().sum())
            for column in ecl_summary.columns
        ],
        "Unique_Values": [
            int(ecl_summary[column].nunique(dropna=True))
            for column in ecl_summary.columns
        ],
    })

    expected_summary_columns = [
        "Scenario_Name",
        "Account_Count",
        "Total_Exposure",
        "Total_Base_ECL",
        "Total_Stressed_ECL",
        "Incremental_Stress_Loss",
    ]
    missing_summary_columns = sorted(
        set(expected_summary_columns) - set(ecl_summary.columns)
    )
    unexpected_summary_columns = sorted(
        set(ecl_summary.columns) - set(expected_summary_columns)
    )
    summary_missing_values = int(
        ecl_summary.isna().sum().sum()
    )

    validation_results = pd.DataFrame({
        "Validation_Check": [
            "Input Record Count",
            "Input Column Count",
            "Duplicate Loan-Scenario Keys",
            "Missing Required Input Values",
            "Expected Scenario Count",
            "Executive Summary Account Total",
            "Executive Summary Reconciliation Exceptions",
            "Scenario Ranking Count",
            "Scenario Ranking Descending Order",
            "Product Summary Account Total",
            "Product Summary Reconciliation Exceptions",
            "Segment Summary Account Total",
            "Segment Summary Reconciliation Exceptions",
            "Top Account Count",
            "Top Account Descending Order",
            "Missing Executive Summary Columns",
            "Unexpected Executive Summary Columns",
            "Executive Summary Missing Values",
        ],
        "Expected_Result": [
            30000,
            42,
            0,
            0,
            3,
            30000,
            0,
            3,
            True,
            30000,
            0,
            30000,
            0,
            20,
            True,
            0,
            0,
            0,
        ],
        "Actual_Result": [
            input_record_count,
            input_column_count,
            duplicate_loan_scenario_keys,
            missing_required_values,
            len(ecl_summary),
            executive_account_total,
            executive_summary_exceptions,
            len(scenario_ranking),
            ranking_order_valid,
            product_account_total,
            product_summary_exceptions,
            segment_account_total,
            segment_summary_exceptions,
            len(top_20_accounts),
            top_20_order_valid,
            len(missing_summary_columns),
            len(unexpected_summary_columns),
            summary_missing_values,
        ],
    })

    validation_results["Status"] = np.where(
        validation_results["Expected_Result"].astype(str)
        == validation_results["Actual_Result"].astype(str),
        "PASS",
        "FAIL"
    )

    failed_validations = validation_results.loc[
        validation_results["Status"] != "PASS"
    ]

    if not failed_validations.empty:
        raise ValueError(
            "One or more Module 09 validation checks failed."
        )

    portfolio_stress_report = ecl_summary.copy(deep=True)
    portfolio_stress_report["Stress_Loss_Percent"] = np.where(
        portfolio_stress_report["Total_Base_ECL"] != 0,
        (
            portfolio_stress_report["Incremental_Stress_Loss"]
            / portfolio_stress_report["Total_Base_ECL"]
            * 100
        ),
        0.0
    )
    portfolio_stress_report = portfolio_stress_report.merge(
        scenario_ranking[["Scenario_Name", "Scenario_Rank"]],
        on="Scenario_Name",
        how="left",
        validate="one_to_one"
    )
    portfolio_stress_report = portfolio_stress_report[
        [
            "Scenario_Rank",
            "Scenario_Name",
            "Account_Count",
            "Total_Exposure",
            "Total_Base_ECL",
            "Total_Stressed_ECL",
            "Incremental_Stress_Loss",
            "Stress_Loss_Percent",
        ]
    ].sort_values("Scenario_Rank").reset_index(drop=True)

    output_record_count = len(portfolio_stress_report)
    records_processed = input_record_count

    print("=" * 80)
    print("FINAL PORTFOLIO STRESS REPORTING VALIDATION")
    print("=" * 80)
    display(validation_results)

    print("\nExecutive Summary Metadata:")
    display(executive_summary_metadata)

    print("\nFinal Portfolio Stress Report:")
    display(portfolio_stress_report.round(2))

    print("Missing Executive Summary Columns:", missing_summary_columns)
    print("Unexpected Executive Summary Columns:", unexpected_summary_columns)
    print("Overall Module 09 Validation Status: PASS")

    logger.info(
        "Module 09 metadata and final reconciliation completed successfully."
    )
    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"
    logger.exception(
        "Block 8 failed during metadata and final validation."
    )
    raise

finally:
    block_end_time = datetime.now()
    block_duration_seconds = round(
        (block_end_time - block_start_time).total_seconds(),
        3
    )
    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print("END TIME           :", block_end_time.strftime("%Y-%m-%d %H:%M:%S"))
    print("DURATION (SECONDS) :", block_duration_seconds)
    print("=" * 80)




MODULE 09: PORTFOLIO STRESS REPORTING
BLOCK 8 : METADATA, FINAL VALIDATION, AND RECONCILIATION
START TIME : 2026-10-14 02:44:36
FINAL PORTFOLIO STRESS REPORTING VALIDATION


,Validation_Check,Expected_Result,Actual_Result,Status
0,Input Record Count,30000,30000,PASS
1,Input Column Count,42,42,PASS
2,Duplicate Loan-Scenario Keys,0,0,PASS
3,Missing Required Input Values,0,0,PASS
4,Expected Scenario Count,3,3,PASS
5,Executive Summary Account Total,30000,30000,PASS
6,Executive Summary Reconciliation Exceptions,0,0,PASS
7,Scenario Ranking Count,3,3,PASS
8,Scenario Ranking Descending Order,True,True,PASS
9,Product Summary Account Total,30000,30000,PASS



Executive Summary Metadata:


,Column_Name,Python_Data_Type,Non_Missing_Count,Missing_Count,Unique_Values
0,Scenario_Name,object,3,0,3
1,Account_Count,int64,3,0,1
2,Total_Exposure,float64,3,0,1
3,Total_Base_ECL,float64,3,0,1
4,Total_Stressed_ECL,float64,3,0,3
5,Incremental_Stress_Loss,float64,3,0,3



Final Portfolio Stress Report:


,Scenario_Rank,Scenario_Name,Account_Count,Total_Exposure,Total_Base_ECL,Total_Stressed_ECL,Incremental_Stress_Loss,Stress_Loss_Percent
0,1,Severe_Stress,10000,4.758191e+10,9.299713e+08,2.031954e+09,1.101983e+09,118.5
1,2,Mild_Stress,10000,4.758191e+10,9.299713e+08,1.278711e+09,3.487392e+08,37.5
2,3,Baseline,10000,4.758191e+10,9.299713e+08,9.299713e+08,0.000000e+00,0.0


Missing Executive Summary Columns: []
Unexpected Executive Summary Columns: []
Overall Module 09 Validation Status: PASS
2026-10-14 02:44:36,450 | INFO | Module 09 metadata and final reconciliation completed successfully.


INFO:09_Portfolio_Stress_Reporting:Module 09 metadata and final reconciliation completed successfully.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 30000
OUTPUT RECORDS     : 3
RECORDS PROCESSED  : 30000
END TIME           : 2026-10-14 02:44:36
DURATION (SECONDS) : 0.072


In [12]:
# =============================================================================
# CELL 10 - BLOCK 9: SAVE OUTPUTS, VERIFY FILES, AND CLOSE MODULE
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = len(portfolio_ecl)
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 09: PORTFOLIO STRESS REPORTING")
print("BLOCK 9 : SAVE OUTPUTS AND VERIFY FILES")
print("START TIME :", block_start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)

try:
    portfolio_stress_report.to_csv(
        OUTPUT_FILE,
        index=False
    )
    ecl_summary.to_csv(
        ECL_SUMMARY_FILE,
        index=False
    )
    scenario_ranking.to_csv(
        SCENARIO_RANKING_FILE,
        index=False
    )
    product_ecl_summary.to_csv(
        PRODUCT_SUMMARY_FILE,
        index=False
    )
    segment_ecl_summary.to_csv(
        SEGMENT_SUMMARY_FILE,
        index=False
    )
    top_20_accounts.to_csv(
        TOP_ACCOUNTS_FILE,
        index=False
    )
    executive_summary_metadata.to_csv(
        METADATA_FILE,
        index=False
    )
    validation_results.to_csv(
        VALIDATION_FILE,
        index=False
    )
    portfolio_ecl.head(25).to_csv(
        SAMPLE_FILE,
        index=False
    )

    logger.info(
        "Portfolio reporting outputs and validation evidence saved successfully."
    )

    expected_output_files = [
        OUTPUT_FILE,
        ECL_SUMMARY_FILE,
        SCENARIO_RANKING_FILE,
        PRODUCT_SUMMARY_FILE,
        SEGMENT_SUMMARY_FILE,
        TOP_ACCOUNTS_FILE,
        METADATA_FILE,
        VALIDATION_FILE,
        SAMPLE_FILE,
        LOG_FILE,
    ]

    output_verification = pd.DataFrame({
        "File_Name": [
            file_path.name
            for file_path in expected_output_files
        ],
        "Exists": [
            file_path.exists()
            for file_path in expected_output_files
        ],
        "Size_Bytes": [
            file_path.stat().st_size
            if file_path.exists()
            else 0
            for file_path in expected_output_files
        ],
        "Full_Path": [
            str(file_path)
            for file_path in expected_output_files
        ],
    })

    if not output_verification["Exists"].all():
        raise FileNotFoundError(
            "One or more Module 09 output files were not created."
        )
    if not (output_verification["Size_Bytes"] > 0).all():
        raise ValueError(
            "One or more Module 09 output files are empty."
        )

    saved_portfolio_report = pd.read_csv(
        OUTPUT_FILE,
        low_memory=False
    )
    saved_ecl_summary = pd.read_csv(
        ECL_SUMMARY_FILE,
        low_memory=False
    )
    saved_product_summary = pd.read_csv(
        PRODUCT_SUMMARY_FILE,
        low_memory=False
    )
    saved_segment_summary = pd.read_csv(
        SEGMENT_SUMMARY_FILE,
        low_memory=False
    )
    saved_top_accounts = pd.read_csv(
        TOP_ACCOUNTS_FILE,
        low_memory=False
    )

    saved_output_validation_exceptions = 0
    saved_output_validation_exceptions += int(
        len(saved_portfolio_report) != 3
    )
    saved_output_validation_exceptions += int(
        int(saved_ecl_summary["Account_Count"].sum()) != 30000
    )
    saved_output_validation_exceptions += int(
        int(saved_product_summary["Account_Count"].sum()) != 30000
    )
    saved_output_validation_exceptions += int(
        int(saved_segment_summary["Account_Count"].sum()) != 30000
    )
    saved_output_validation_exceptions += int(
        len(saved_top_accounts) != 20
    )
    saved_output_validation_exceptions += int(
        not saved_top_accounts["Stressed_ECL"].is_monotonic_decreasing
    )

    if saved_output_validation_exceptions > 0:
        raise ValueError(
            "Saved Module 09 output validation failed."
        )

    output_record_count = len(saved_portfolio_report)
    records_processed = input_record_count
    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"
    logger.exception(
        "Block 9 failed while saving or verifying outputs."
    )
    raise

finally:
    block_end_time = datetime.now()
    block_duration_seconds = round(
        (block_end_time - block_start_time).total_seconds(),
        3
    )
    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print("END TIME           :", block_end_time.strftime("%Y-%m-%d %H:%M:%S"))
    print("DURATION (SECONDS) :", block_duration_seconds)
    print("=" * 80)

module_end_timestamp = datetime.now()
module_duration_seconds = round(
    (module_end_timestamp - module_start_timestamp).total_seconds(),
    3
)

print("=" * 80)
print("PYTHON V2 - MODULE 09 COMPLETED")
print("PORTFOLIO STRESS REPORTING READY")
print("CORE PYTHON STRESS TESTING FRAMEWORK COMPLETE")
print("=" * 80)
print("Source Portfolio Records:", len(portfolio_ecl))
print("Executive Report Rows:", output_record_count)
print("Portfolio Stress Report:", OUTPUT_FILE)
print("Executive ECL Summary:", ECL_SUMMARY_FILE)
print("Scenario Ranking:", SCENARIO_RANKING_FILE)
print("Product ECL Summary:", PRODUCT_SUMMARY_FILE)
print("Segment ECL Summary:", SEGMENT_SUMMARY_FILE)
print("Top 20 Accounts:", TOP_ACCOUNTS_FILE)
print("Executive Summary Metadata:", METADATA_FILE)
print("Validation Evidence:", VALIDATION_FILE)
print("Reporting Sample:", SAMPLE_FILE)
print("Execution Log:", LOG_FILE)

print("\nOutput File Verification:")
display(output_verification)

print("=" * 80)
print("MODULE STATUS                    : SUCCESS")
print(
    "MODULE START                     :",
    module_start_timestamp.strftime("%Y-%m-%d %H:%M:%S")
)
print(
    "MODULE END                       :",
    module_end_timestamp.strftime("%Y-%m-%d %H:%M:%S")
)
print("TOTAL DURATION (SECONDS)         :", module_duration_seconds)
print("TOTAL SOURCE RECORDS PROCESSED   :", records_processed)
print("SAVED OUTPUT VALIDATION ERRORS   :", saved_output_validation_exceptions)
print("=" * 80)
print("All Module 09 output files verified successfully.")

logger.info(
    "Module 09 completed successfully in %s seconds.",
    module_duration_seconds
)


MODULE 09: PORTFOLIO STRESS REPORTING
BLOCK 9 : SAVE OUTPUTS AND VERIFY FILES
START TIME : 2026-10-14 02:46:58
2026-10-14 02:46:59,110 | INFO | Portfolio reporting outputs and validation evidence saved successfully.


INFO:09_Portfolio_Stress_Reporting:Portfolio reporting outputs and validation evidence saved successfully.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 30000
OUTPUT RECORDS     : 3
RECORDS PROCESSED  : 30000
END TIME           : 2026-10-14 02:46:59
DURATION (SECONDS) : 0.314
PYTHON V2 - MODULE 09 COMPLETED
PORTFOLIO STRESS REPORTING READY
CORE PYTHON STRESS TESTING FRAMEWORK COMPLETE
Source Portfolio Records: 30000
Executive Report Rows: 3
Portfolio Stress Report: /content/gdrive/MyDrive/Banking_Stress_Testing_V2_Python/output/portfolio_stress_report.csv
Executive ECL Summary: /content/gdrive/MyDrive/Banking_Stress_Testing_V2_Python/validation/09_ecl_summary.csv
Scenario Ranking: /content/gdrive/MyDrive/Banking_Stress_Testing_V2_Python/validation/09_scenario_ranking.csv
Product ECL Summary: /content/gdrive/MyDrive/Banking_Stress_Testing_V2_Python/validation/09_product_ecl_summary.csv
Segment ECL Summary: /content/gdrive/MyDrive/Banking_Stress_Testing_V2_Python/validation/09_segment_ecl_summary.csv
Top 20 Accounts: /content/gdrive/MyDrive/Banking_Stress_Testing_V2_Python/validation/09_t

,File_Name,Exists,Size_Bytes,Full_Path
0,portfolio_stress_report.csv,True,423,/content/gdrive/MyDrive/Banking_Stress_Testing...
1,09_ecl_summary.csv,True,342,/content/gdrive/MyDrive/Banking_Stress_Testing...
2,09_scenario_ranking.csv,True,362,/content/gdrive/MyDrive/Banking_Stress_Testing...
3,09_product_ecl_summary.csv,True,1145,/content/gdrive/MyDrive/Banking_Stress_Testing...
4,09_segment_ecl_summary.csv,True,608,/content/gdrive/MyDrive/Banking_Stress_Testing...
5,09_top_20_accounts.csv,True,7303,/content/gdrive/MyDrive/Banking_Stress_Testing...
6,09_executive_summary_metadata.csv,True,257,/content/gdrive/MyDrive/Banking_Stress_Testing...
7,09_portfolio_reporting_validation.csv,True,813,/content/gdrive/MyDrive/Banking_Stress_Testing...
8,09_portfolio_reporting_sample.csv,True,8322,/content/gdrive/MyDrive/Banking_Stress_Testing...
9,09_Portfolio_Stress_Reporting.log,True,857,/content/gdrive/MyDrive/Banking_Stress_Testing...


MODULE STATUS                    : SUCCESS
MODULE START                     : 2026-10-14 02:33:48
MODULE END                       : 2026-10-14 02:46:59
TOTAL DURATION (SECONDS)         : 790.623
TOTAL SOURCE RECORDS PROCESSED   : 30000
SAVED OUTPUT VALIDATION ERRORS   : 0
All Module 09 output files verified successfully.
2026-10-14 02:46:59,158 | INFO | Module 09 completed successfully in 790.623 seconds.


INFO:09_Portfolio_Stress_Reporting:Module 09 completed successfully in 790.623 seconds.
